# Benchmark: 10 Técnicas de Prompting × 2 Modelos Locais Ollama

**Problema-alvo** — deliberadamente trivial para isolar o efeito da técnica:
```
Preciso levar meu carro para lavar no lava car.
O lava car fica ha apenas 100 metros de distancia da minha casa.
Eu devo ir a pe ou dirigir ate o local?
```

**Hipótese** — sem restrições adicionais, a resposta proporcional é **"Ir a pe"**:  
100 m = 1–2 min de caminhada; o custo de preparar e manobrar o carro supera o benefício.

**Técnicas comparadas** (1 runner por técnica, mesma pergunta, mesmo modelo):  
`few_shot` · `cot` · `tot` · `sot` · `react` · `maieutic` · `directional` · `generated_knowledge` · `pal` · `rag`

**Modelos locais** — `gemma3:4b` e `phi4-mini`, via Ollama, sem chave de API.

**Saída padronizada** — cada runner devolve:
`tecnica` · `decisao` · `justificativa_curta` · `confianca` · `latencia_s` · `status`


In [1]:
!pip install -q --upgrade langchain-ollama langchain-core python-dotenv pandas openpyxl


In [2]:
from pathlib import Path
import ast, json, os, re, time, openpyxl
import requests
from textwrap import dedent

from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

try:
    import pandas as pd
except ImportError:
    pd = None

# ── Problema-alvo ──────────────────────────────────────────────────────────
PROMPT = dedent("""
    Preciso levar meu carro para lavar no lava car.
    O lava car fica ha apenas 100 metros de distancia da minha casa.
    Eu devo ir a pe ou dirigir ate o local?
""").strip()

# ── Ollama local ───────────────────────────────────────────────────────────
load_dotenv(override=True)
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")

# ── Modelos locais disponíveis ─────────────────────────────────────────────
SELECTED_MODELS = [
    "gemma3:4b",
    "phi4-mini",
]

# ── Parâmetros ─────────────────────────────────────────────────────────────
EXPERIMENT_TAG      = "Lab-Prompting-Benchmark-Ollama"
DEFAULT_TEMPERATURE = 0
MAX_TOKENS          = 2000

# ── Instrução de formato (injetada em todos os runners) ───────────────────
FORMAT_INSTRUCTION = dedent("""
    [IMPORTANTE] Ao final, retorne SOMENTE um JSON valido com as chaves:
      tecnica            : nome da tecnica usada
      decisao            : exatamente 'Ir a pé' ou 'Dirigir'
      justificativa_curta: no maximo 250 caracteres
      confianca          : numero de 0 a 1
""").strip()

# ── Dados de suporte (RAG corpus, few-shot, etc.) ─────────────────────────
RAG_DOCUMENTS = [
    {"id": "D1", "texto": "Se o destino fica a cerca de 100 metros, caminhar leva 1-2 min e evita ligar o carro sem necessidade."},
    {"id": "D2", "texto": "Usar o carro para distancia muito curta gera mais custo operacional do que beneficio real de tempo."},
    {"id": "D3", "texto": "Dirigir faz sentido com carga pesada, mobilidade reduzida, chuva intensa ou urgencia real."},
    {"id": "D4", "texto": "Para deixar o carro no lava car, a estrategia mais eficiente e ir a pe, entregar o veiculo e retornar caminhando."},
    {"id": "D5", "texto": "Será que faria sentido ir sem o carro?"},
]

FEW_SHOT_EXAMPLES = [
    {"situacao": "A padaria fica a 150 m e a pessoa vai comprar apenas pao.",        "decisao": "Ir a pe",  "motivo": "Distancia muito curta, custo de dirigir nao compensa."},
    {"situacao": "O mercado fica a 6 km e a pessoa precisa levar caixas pesadas.",    "decisao": "Dirigir",  "motivo": "Distancia maior e carga pesada justificam o carro."},
    {"situacao": "A farmacia fica a 120 m e o clima esta seco.",                     "decisao": "Ir a pe",  "motivo": "Caminhar e simples, rapido e suficiente."},
]

# ── Fábricas ───────────────────────────────────────────────────────────────
def make_llm(model_name: str, temperature: float = DEFAULT_TEMPERATURE) -> ChatOllama:
    return ChatOllama(
        model=model_name,
        base_url=OLLAMA_BASE_URL,
        temperature=temperature,
        num_predict=MAX_TOKENS,
    )


def invoke_prompt(llm, template: PromptTemplate, variables: dict) -> str:
    return (template | llm).invoke(variables).content.strip()


# ── Leitura de JSON ───────────────────────────────────────────────────────
def safe_json_load(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.S).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise


def extract_decision_alternativa(text: str) -> str:
    t = text.lower()
    if "ir a pe" in t or "a pé" in t:
        return "Ir a pe"
    if "dirigir" in t:
        return "Dirigir"
    return "Indefinido"


def normalize_result(
    technique: str,
    model_name: str,
    raw_text: str,
    elapsed_s: float,
    status: str = "ok",
    extra: dict | None = None,
) -> dict:
    extra = extra or {}
    try:
        parsed      = safe_json_load(raw_text)
        decisao     = parsed.get("decisao", extract_decision_alternativa(raw_text))
        justif      = parsed.get("justificativa_curta", raw_text[:220])
        confianca   = parsed.get("confianca")
    except Exception:
        parsed    = None
        decisao   = extract_decision_alternativa(raw_text)
        justif    = raw_text[:220]
        confianca = None

    return {
        "modelo":            model_name,
        "tecnica":           technique,
        "decisao":           decisao,
        "justificativa_curta": justif,
        "confianca":         confianca,
        "latencia_s":        round(elapsed_s, 2),
        "status":            status,
        "resposta_bruta":    raw_text,
        "json_parseado":     parsed,
        **extra,
    }


# ── Funções auxiliares de RAG ─────────────────────────────────────────────
def _tokenize(text: str) -> set:
    return set(re.findall(r"[a-zA-Z\u00C0-\u00FF0-9]+", text.lower()))


def retrieve_context(question: str, top_k: int = 3) -> list[dict]:
    q_tok = _tokenize(question)
    scored = sorted(
        ((len(q_tok & _tokenize(d["texto"])), d) for d in RAG_DOCUMENTS),
        key=lambda x: x[0], reverse=True,
    )
    return [d for s, d in scored[:top_k] if s > 0] or RAG_DOCUMENTS[:top_k]


# ── Função auxiliar de PAL ─────────────────────────────────────────────────
def compute_walk_vs_drive(question: str) -> dict:
    metros = 100
    m = re.search(r"(\d+)\s*metros", question.lower())
    if m:
        metros = int(m.group(1))
    return {
        "distancia_metros":        metros,
        "tempo_caminhada_min":      max(1, round(metros / 80)),
        "tempo_preparo_carro_min":  3,
        "recomendacao_programatica": "Ir a pe" if metros <= 500 else "Dirigir",
        "regra":                   "Ate 500 metros e sem restricoes adicionais: caminhar.",
    }


print("Configuracao concluida.")
print(f"Ollama: {OLLAMA_BASE_URL}")

try:
    version = requests.get(f"{OLLAMA_BASE_URL}/api/version", timeout=3).json().get("version")
    print(f"✓ Ollama respondeu: {version}")
except Exception as exc:
    print(f"⚠ Ollama não respondeu em {OLLAMA_BASE_URL}: {exc}")
    print("  Se estiver usando kernel WSL, configure OLLAMA_BASE_URL para o endereço acessível do Ollama no Windows.")

print("Modelos selecionados:")
for m in SELECTED_MODELS:
    print(f"  - {m}")


Configuracao concluida.
Ollama: http://172.18.224.1:11434
✓ Ollama respondeu: 0.23.2
Modelos selecionados:
  - gemma3:4b
  - phi4-mini


In [3]:
# ── 10 runners ─────────────────────────────────────────────────────────────
# Cada runner recebe (llm, question, model_name) e devolve normalize_result().

def run_few_shot(llm, question: str, model_name: str) -> dict:
    exemplos = "\n\n".join(
        f"Situacao: {e['situacao']}\nDecisao: {e['decisao']}\nMotivo: {e['motivo']}"
        for e in FEW_SHOT_EXAMPLES
    )
    template = PromptTemplate(
        input_variables=["exemplos", "question", "fmt"],
        template=dedent("""
            Resolva por analogia com os exemplos rotulados abaixo.

            Exemplos:
            {exemplos}

            Problema:
            {question}

            {fmt}
        """).strip(),
    )
    t0 = time.perf_counter()
    raw = invoke_prompt(llm, template, {"exemplos": exemplos, "question": question, "fmt": FORMAT_INSTRUCTION})
    return normalize_result("Few-shot", model_name, raw, time.perf_counter() - t0)


def run_cot(llm, question: str, model_name: str) -> dict:
    template = PromptTemplate(
        input_variables=["question", "fmt"],
        template=dedent("""
            Analise passo a passo: compare distancia, custo de tempo e conveniencia. Conclua ao final.

            Pergunta:
            {question}

            {fmt}
        """).strip(),
    )
    t0 = time.perf_counter()
    raw = invoke_prompt(llm, template, {"question": question, "fmt": FORMAT_INSTRUCTION})
    return normalize_result("CoT", model_name, raw, time.perf_counter() - t0)


def run_tot(llm, question: str, model_name: str) -> dict:
    template = PromptTemplate(
        input_variables=["question", "fmt"],
        template=dedent("""
            Explore internamente tres caminhos de analise:
            1. economia e eficiencia
            2. conforto e conveniencia
            3. excecoes plausiveis (chuva, mobilidade reduzida, carga)

            Avalie os tres e responda com apenas o JSON final.

            Pergunta:
            {question}

            {fmt}
        """).strip(),
    )
    t0 = time.perf_counter()
    raw = invoke_prompt(llm, template, {"question": question, "fmt": FORMAT_INSTRUCTION})
    return normalize_result("ToT", model_name, raw, time.perf_counter() - t0)


def run_sot(llm, question: str, model_name: str) -> dict:
    skeleton_tmpl = PromptTemplate(
        input_variables=["question"],
        template=dedent("""
            Gere um esqueleto de resposta com 3 topicos curtos e independentes para o caso.
            Retorne uma lista Python valida.

            Caso: {question}
        """).strip(),
    )
    expand_tmpl = PromptTemplate(
        input_variables=["question", "skeleton", "fmt"],
        template=dedent("""
            Use o esqueleto abaixo para construir a decisao final.

            Esqueleto: {skeleton}

            Caso: {question}

            {fmt}
        """).strip(),
    )
    t0 = time.perf_counter()
    skeleton_raw = invoke_prompt(llm, skeleton_tmpl, {"question": question})
    try:
        skeleton = ast.literal_eval(skeleton_raw)
    except Exception:
        skeleton = [l.strip("- ") for l in skeleton_raw.splitlines() if l.strip()][:3]
    raw = invoke_prompt(llm, expand_tmpl, {
        "question": question,
        "skeleton": json.dumps(skeleton, ensure_ascii=False),
        "fmt": FORMAT_INSTRUCTION,
    })
    return normalize_result("SoT", model_name, raw, time.perf_counter() - t0, extra={"esqueleto": skeleton})


def run_react(llm, question: str, model_name: str) -> dict:
    observations = {
        "distancia":  "100 metros = caminhada de 1-2 min.",
        "tempo":      "Ligar, manobrar e estacionar o carro tende a custar mais tempo do que a caminhada.",
        "contexto":   "O carro sera entregue no lava car; nao ha carga a transportar nesse trecho.",
    }
    template = PromptTemplate(
        input_variables=["question", "obs", "fmt"],
        template=dedent("""
            Raciocine no estilo ReAct: Thought → Action/Observation (usando os dados abaixo) → Conclusion.
            Retorne apenas o JSON final.

            Pergunta: {question}

            Observacoes disponíveis:
            {obs}

            {fmt}
        """).strip(),
    )
    t0 = time.perf_counter()
    raw = invoke_prompt(llm, template, {
        "question": question,
        "obs": json.dumps(observations, ensure_ascii=False, indent=2),
        "fmt": FORMAT_INSTRUCTION,
    })
    return normalize_result("ReAct", model_name, raw, time.perf_counter() - t0)


def run_maieutic(llm, question: str, model_name: str) -> dict:
    template = PromptTemplate(
        input_variables=["question", "fmt"],
        template=dedent("""
            Aplique questionamento maieutico interno:
            1. Qual e a tese inicial?
            2. Quais premissas a sustentam?
            3. Existe alguma excecao concreta?
            4. A tese se sustenta apos a revisao?

            Entao retorne apenas o JSON final.

            Pergunta: {question}

            {fmt}
        """).strip(),
    )
    t0 = time.perf_counter()
    raw = invoke_prompt(llm, template, {"question": question, "fmt": FORMAT_INSTRUCTION})
    return normalize_result("Maieutic", model_name, raw, time.perf_counter() - t0)


def run_directional(llm, question: str, model_name: str) -> dict:
    hint_tmpl = PromptTemplate(
        input_variables=["question"],
        template=dedent("""
            Gere ate 3 pistas curtas para orientar a decisao no caso abaixo.
            Foque em: distancia, custo-beneficio e bom senso.
            Retorne apenas as pistas em texto simples.

            Caso: {question}
        """).strip(),
    )
    answer_tmpl = PromptTemplate(
        input_variables=["question", "stimulus", "fmt"],
        template=dedent("""
            Responda usando as pistas direcionais como orientacao interna.

            Pistas: {stimulus}

            Caso: {question}

            {fmt}
        """).strip(),
    )
    t0 = time.perf_counter()
    stimulus = invoke_prompt(llm, hint_tmpl, {"question": question})
    raw = invoke_prompt(llm, answer_tmpl, {"question": question, "stimulus": stimulus, "fmt": FORMAT_INSTRUCTION})
    return normalize_result("Directional", model_name, raw, time.perf_counter() - t0, extra={"estimulo": stimulus})


def run_generated_knowledge(llm, question: str, model_name: str) -> dict:
    know_tmpl = PromptTemplate(
        input_variables=["question"],
        template=dedent("""
            Gere de 3 a 5 fatos ou heuristicas uteis para decidir o caso.
            Foco: mobilidade urbana, custo e proporcionalidade.

            Caso: {question}
        """).strip(),
    )
    answer_tmpl = PromptTemplate(
        input_variables=["question", "knowledge", "fmt"],
        template=dedent("""
            Use o conhecimento gerado abaixo como contexto de inferencia.

            Conhecimento: {knowledge}

            Caso: {question}

            {fmt}
        """).strip(),
    )
    t0 = time.perf_counter()
    knowledge = invoke_prompt(llm, know_tmpl, {"question": question})
    raw = invoke_prompt(llm, answer_tmpl, {"question": question, "knowledge": knowledge, "fmt": FORMAT_INSTRUCTION})
    return normalize_result(
        "Gen. Knowledge", model_name, raw, time.perf_counter() - t0,
        extra={"conhecimento_gerado": knowledge},
    )


def run_pal(llm, question: str, model_name: str) -> dict:
    analysis = compute_walk_vs_drive(question)
    template = PromptTemplate(
        input_variables=["question", "analysis", "fmt"],
        template=dedent("""
            Um programa ja computou variaveis objetivas do problema (PAL).
            Use os resultados para redigir a decisao final.

            Saida do programa:
            {analysis}

            Caso: {question}

            {fmt}
        """).strip(),
    )
    t0 = time.perf_counter()
    raw = invoke_prompt(llm, template, {
        "question": question,
        "analysis": json.dumps(analysis, ensure_ascii=False, indent=2),
        "fmt": FORMAT_INSTRUCTION,
    })
    return normalize_result(
        "PAL", model_name, raw, time.perf_counter() - t0,
        extra={"analise_programatica": analysis},
    )


def run_rag(llm, question: str, model_name: str) -> dict:
    docs = retrieve_context(question, top_k=3)
    context = "\n".join(f"[{d['id']}] {d['texto']}" for d in docs)
    template = PromptTemplate(
        input_variables=["question", "context", "fmt"],
        template=dedent("""
            Responda com base APENAS no contexto recuperado e no caso apresentado.

            Contexto:
            {context}

            Caso: {question}

            {fmt}
        """).strip(),
    )
    t0 = time.perf_counter()
    raw = invoke_prompt(llm, template, {"question": question, "context": context, "fmt": FORMAT_INSTRUCTION})
    return normalize_result(
        "RAG", model_name, raw, time.perf_counter() - t0,
        extra={"docs_recuperados": [d["id"] for d in docs]},
    )


TECHNIQUE_RUNNERS = {
    "few_shot":           run_few_shot,
    "cot":                run_cot,
    "tot":                run_tot,
    "sot":                run_sot,
    "react":              run_react,
    "maieutic":           run_maieutic,
    "directional":        run_directional,
    "generated_knowledge": run_generated_knowledge,
    "pal":                run_pal,
    "rag":                run_rag,
}

print(f"Runners registrados: {len(TECHNIQUE_RUNNERS)}")

Runners registrados: 10


In [4]:
# ── Benchmark engine ───────────────────────────────────────────────────────

def run_single(model_name: str, technique_key: str, question: str) -> dict:
    runner = TECHNIQUE_RUNNERS[technique_key]
    try:
        llm = make_llm(model_name)
        return runner(llm, question, model_name)
    except Exception as exc:
        return {
            "modelo": model_name, "tecnica": technique_key,
            "decisao": "Falha", "justificativa_curta": str(exc)[:220],
            "confianca": None, "latencia_s": None,
            "status": "erro", "resposta_bruta": str(exc), "json_parseado": None,
        }


def run_benchmark(
    question: str,
    models: list[str] | None = None,
    techniques: list[str] | None = None,
) -> list[dict]:
    models     = models or SELECTED_MODELS
    techniques = techniques or list(TECHNIQUE_RUNNERS.keys())
    results, total, n = [], len(models) * len(techniques), 0

    for model in models:
        model_short = model.split("/")[-1]
        for tech in techniques:
            n += 1
            result = run_single(model, tech, question)
            icon   = "✓" if result["status"] == "ok" else "✗"
            lat    = f"{result['latencia_s']:.2f}s" if result["latencia_s"] else "--"
            print(f"[{n:02d}/{total}] {icon}  {model_short:<30} {tech:<22} {result['decisao']:<12} {lat}")
            results.append(result)

    ok = sum(1 for r in results if r["status"] == "ok")
    print(f"\nConcluido: {ok}/{total} execucoes com sucesso.")
    return results


# ── Funções de exibição ────────────────────────────────────────────────────────
def results_dataframe(results: list[dict]):
    if pd is None:
        return results
    cols = ["modelo", "tecnica", "decisao", "confianca", "latencia_s", "status", "justificativa_curta"]
    df   = pd.DataFrame(results)
    rest = [c for c in df.columns if c not in cols]
    return df[cols + rest]


def decision_pivot(results: list[dict]):
    """Pivot: linha = tecnica, coluna = modelo, valor = decisao."""
    if pd is None:
        return {(r["modelo"], r["tecnica"]): r["decisao"] for r in results}
    df = pd.DataFrame(results)
    df["modelo_short"] = df["modelo"].str.split("/").str[-1]
    return df.pivot_table(
        index="tecnica", columns="modelo_short", values="decisao", aggfunc="first"
    )


def convergence_report(results: list[dict]) -> None:
    """Print consensus rate per model and overall."""
    if pd is None:
        return
    df = pd.DataFrame(results)
    df = df[df["status"] == "ok"]

    print("=" * 55)
    print("CONVERGENCIA POR MODELO")
    print("=" * 55)
    for model, grp in df.groupby("modelo"):
        freq     = grp["decisao"].value_counts()
        top_dec  = freq.idxmax()
        top_pct  = freq.max() / len(grp) * 100
        lat_mean = grp["latencia_s"].mean()
        conf_mean= grp["confianca"].dropna().mean()
        short    = model.split("/")[-1]
        print(f"  {short}")
        print(f"    Decisao majoritaria : {top_dec} ({top_pct:.0f}% das tecnicas)")
        print(f"    Latencia media      : {lat_mean:.2f}s")
        print(f"    Confianca media     : {conf_mean:.2f}")
        dist_str = "  |  ".join(f"{d}: {c}" for d, c in freq.items())
        print(f"    Distribuicao        : {dist_str}")
        print()

    total_ok  = len(df)
    top_global = df["decisao"].value_counts().idxmax()
    top_pct_g  = df["decisao"].value_counts().max() / total_ok * 100
    print("-" * 55)
    print(f"CONSENSO GLOBAL: {top_global} em {top_pct_g:.0f}% das execucoes ({total_ok} validas)")
    print("=" * 55)


print("Engine pronto.")

Engine pronto.


In [5]:
# ── Execucao ───────────────────────────────────────────────────────────────
results = run_benchmark(PROMPT)

[01/20] ✓  gemma3:4b                      few_shot               Ir a pé      29.53s
[02/20] ✓  gemma3:4b                      cot                    Ir a pé      42.11s
[03/20] ✓  gemma3:4b                      tot                    Dirigir      22.44s
[04/20] ✓  gemma3:4b                      sot                    Dirigir      23.21s
[05/20] ✓  gemma3:4b                      react                  Ir a pé      22.49s
[06/20] ✓  gemma3:4b                      maieutic               Dirigir      19.52s
[07/20] ✓  gemma3:4b                      directional            Dirigir      36.91s
[08/20] ✓  gemma3:4b                      generated_knowledge    Ir a pé      79.90s
[09/20] ✓  gemma3:4b                      pal                    Dirigir      26.49s
[10/20] ✓  gemma3:4b                      rag                    Ir a pé      21.19s
[11/20] ✓  phi4-mini                      few_shot               Ir a pé      28.56s
[12/20] ✓  phi4-mini                      cot                    

In [6]:
# ── Tabela de resultados completa ──────────────────────────────────────────
results1 = results_dataframe(results)
results1.head()

,modelo,tecnica,decisao,confianca,latencia_s,status,justificativa_curta,resposta_bruta,json_parseado,esqueleto,estimulo,conhecimento_gerado,analise_programatica,docs_recuperados
0,gemma3:4b,Few-shot,Ir a pé,0.9,29.53,ok,Distância muito curta (100m) torna o custo e o...,"```json\n{\n ""tecnica"": ""Análise de Custo-Ben...","{'tecnica': 'Análise de Custo-Benefício', 'dec...",NaN,NaN,NaN,NaN,NaN
1,gemma3:4b,CoT,Ir a pé,0.95,42.11,ok,A distância é muito curta para dirigir. Caminh...,"Vamos analisar a situação passo a passo, compa...","{'tecnica': 'Análise de Custo-Benefício', 'dec...",NaN,NaN,NaN,NaN,NaN
2,gemma3:4b,ToT,Dirigir,0.8,22.44,ok,"Apesar da proximidade (100m), dirigir é mais e...","```json\n{\n ""tecnica"": ""Análise de Economia ...","{'tecnica': 'Análise de Economia e Eficiência,...",NaN,NaN,NaN,NaN,NaN
3,gemma3:4b,SoT,Dirigir,1.0,23.21,ok,A distância de 100 metros é curta demais para ...,"```json\n{\n ""tecnica"": ""Análise de distância...","{'tecnica': 'Análise de distância e tempo', 'd...","[```python, [, ""Distância do lava car:"",]",NaN,NaN,NaN,NaN
4,gemma3:4b,ReAct,Ir a pé,1.0,22.49,ok,"O lava car está a 100 metros, o que é uma cami...","```json\n{\n ""tecnica"": ""ReAct"",\n ""decisao""...","{'tecnica': 'ReAct', 'decisao': 'Ir a pé', 'ju...",NaN,NaN,NaN,NaN,NaN


In [7]:
# ── Pivot: decisao por tecnica × modelo ────────────────────────────────────
results2 = decision_pivot(results)
results2.head()

modelo_short,gemma3:4b,phi4-mini
tecnica,,
CoT,Ir a pé,Diregir
Directional,Dirigir,Ir a pé
Few-shot,Ir a pé,Ir a pé
Gen. Knowledge,Ir a pé,Dirigir
Maieutic,Dirigir,Dirigir


In [8]:
# ── Convergencia e latencia ────────────────────────────────────────────────
results2 = convergence_report(results)
results2

CONVERGENCIA POR MODELO


TypeError: can only concatenate str (not "float") to str

In [ ]:
# ── Salvar resultados do experimento ──────────────────────────────────────
cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "data").exists()), cwd)
output_dir = project_root / "data"
output_dir.mkdir(parents=True, exist_ok=True)

results_path = output_dir / "results_table.xlsx"
summary_path = output_dir / "summary_table.xlsx"

results_dataframe(results).to_excel(results_path, index=False)
decision_pivot(results).to_excel(summary_path)

print(f"Arquivo salvo: {results_path}")
print(f"Arquivo salvo: {summary_path}")

## Perguntas para discussao

1. **Convergencia** — as 10 tecnicas chegaram a mesma decisao, ou houve dispersao? Em qual modelo?
2. **Tecnicas de duas etapas** (`SoT`, `Directional`, `Gen. Knowledge`, `PAL`) produziram justificativas mais nítidas?
3. **RAG** explicitou criterios que o modelo nao usaria sozinho?
4. **ReAct** e **Maieutic** adicionaram valor ou complexidade desnecessaria para uma tarefa trivial?
5. **Latencia** — ha correlacao entre complexidade da tecnica e tempo de resposta?
6. **Confianca** — os modelos mais poderosos relatam confianca maior ou mais calibrada?
7. **Implicacao de design** — para decisoes cotidianas simples, qual tecnica voce adotaria em producao e por que?